<a href="https://colab.research.google.com/github/NamithaDaney/DataLens-AI/blob/main/DataLens_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas openpyxl

In [5]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI")

INPUT_DIR = BASE_DIR / "input"
REPORT_DIR = BASE_DIR / "reports"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = REPORT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Reports folder:", REPORT_DIR)
print("Plots folder:", PLOTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/input
Reports folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports
Plots folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots


In [94]:
import time

SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls"}
processed_files = set()

print("DataLens AI is watching:", INPUT_DIR)
print("Add a dataset to the input folder to start processing.")
print("Stop this cell using the Colab stop button.")

while True:
    for file_path in INPUT_DIR.iterdir():

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        # Process each filename only once per running session
        if file_path.name in processed_files:
            continue

        try:
            # Wait until the file size is stable before reading it
            size_before = file_path.stat().st_size
            time.sleep(2)
            size_after = file_path.stat().st_size

            if size_before != size_after or size_after == 0:
                continue

            process_dataset(file_path)
            processed_files.add(file_path.name)

        except Exception as error:
            print(f"Could not process {file_path.name}: {error}")

    time.sleep(10)


DataLens AI is watching: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/input
Add a dataset to the input folder to start processing.
Stop this cell using the Colab stop button.

Processing: cardhekho.xlsx
Possible index columns detected: ['Unnamed: 0']
Review these columns before deciding whether to remove them.
Rows: 8369
Columns: 34


KeyboardInterrupt: 

In [7]:
from pathlib import Path
import pandas as pd

def load_dataset(file_path):
    """
    Load a CSV or Excel file into a Pandas DataFrame.
    Detect possible index columns without automatically deleting them.
    Supports .csv, .xlsx and .xls files.
    """

    file_path = Path(file_path)
    extension = file_path.suffix.lower()

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if extension == ".csv":
        df = pd.read_csv(file_path,index_col=0)

    elif extension in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Please upload a CSV or Excel file."
        )

    if df.empty:
        raise ValueError("The uploaded file contains no data.")

    # Detect possible index columns
    unnamed_columns = [
        col for col in df.columns
        if str(col).strip().lower().startswith("unnamed:")
    ]

    if unnamed_columns:
        print("Possible index columns detected:", unnamed_columns)
        print("Review these columns before deciding whether to remove them.")

    return df


In [8]:
def review_unnamed_columns(df):
    """
    Detect possible index columns and recommend an action.
    Does not delete any columns automatically.
    """

    review_results = []

    for col in df.columns:
        column_name = str(col).strip().lower()

        if not column_name.startswith("unnamed:"):
            continue

        values = df[col]

        # Ignore missing values while checking the pattern
        non_missing = values.dropna()

        # Check whether values match 0, 1, 2, 3, ...
        numeric_values = pd.to_numeric(
            non_missing, errors="coerce"
        )

        looks_like_index = (
            len(non_missing) == len(df)
            and len(non_missing) > 0
            and numeric_values.notna().all()
            and numeric_values.reset_index(drop=True).equals(
                pd.Series(range(len(df)), dtype=numeric_values.dtype)
            )
        )

        if looks_like_index:
            recommendation = "Likely exported index — review for removal"
        else:
            recommendation = "Needs manual review — keep for now"

        review_results.append({
            "column": col,
            "missing_values": int(values.isna().sum()),
            "unique_values": int(values.nunique()),
            "sample_values": values.head(5).tolist(),
            "recommendation": recommendation
        })

    return pd.DataFrame(review_results)


In [9]:
def profile_dataset(df):
    """Generate a basic profile for any tabular dataset."""

    profile = {
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": df.columns.tolist(),
        "data_types": {
            col: str(df[col].dtype)
            for col in df.columns
        },
        "missing_values": {
            col: int(df[col].isna().sum())
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "missing_percentage": {
            col: round(
                df[col].isna().mean() * 100, 2
            )
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "duplicate_rows": int(df.duplicated().sum()),
        "unique_values": {
            col: int(df[col].nunique())
            for col in df.columns
        }
    }

    return profile


In [10]:
from itertools import combinations

def find_identical_columns(df):
    """Find exact duplicate columns without changing the dataset."""

    findings = []
    candidate_groups = {}

    # 1. Create a fingerprint for each column
    for position in range(df.shape[1]):
        series = df.iloc[:, position]

        fingerprint = pd.util.hash_pandas_object(
            series, index=False
        ).values.tobytes()

        candidate_groups.setdefault(
            fingerprint, []
        ).append(position)

    # 2. Compare only columns with matching fingerprints
    for positions in candidate_groups.values():
        if len(positions) < 2:
            continue

        for i, j in combinations(positions, 2):
            first = df.iloc[:, i]
            second = df.iloc[:, j]

            # Verify actual equality; never rely on hashes alone
            if first.equals(second):
                findings.append({
                    "column": str(df.columns[i]),
                    "issue": "Identical columns",
                    "count": len(df),
                    "severity": "Medium",
                    "recommendation": (
                        f"Column at position {j} "
                        f"('{df.columns[j]}') is identical to "
                        f"column at position {i} "
                        f"('{df.columns[i]}'). Review before "
                        "removing either column."
                    )
                })

    return findings


In [11]:
import pandas as pd
from itertools import combinations


def find_near_duplicate_columns(df):
    """
    Find text columns that become identical after removing
    leading/trailing spaces and ignoring letter case.
    Does not modify the original DataFrame.
    """
    findings = []
    candidate_groups = {}

    # 1. Normalize text columns only
    for position in range(df.shape[1]):
        series = df.iloc[:, position]

        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        ):
            continue

        normalized = series.map(
            lambda value: (
                value.strip().casefold()
                if isinstance(value, str)
                else value
            )
        )

        # 2. Group potential matches efficiently
        fingerprint = pd.util.hash_pandas_object(
            normalized, index=False
        ).values.tobytes()

        candidate_groups.setdefault(fingerprint, []).append(
            (position, normalized)
        )

    # 3. Verify actual normalized equality
    for group in candidate_groups.values():
        if len(group) < 2:
            continue

        for (i, first), (j, second) in combinations(group, 2):
            if first.equals(second):
                findings.append({
                    "column": str(df.columns[i]),
                    "issue": "Potentially equivalent columns",
                    "count": len(df),
                    "severity": "Low",
                    "recommendation": (
                        f"Column at position {j} "
                        f"('{df.columns[j]}') matches after "
                        "normalizing text case and surrounding "
                        "spaces. Review before merging or removing."
                    )
                })

    return findings

In [62]:
import re
import pandas as pd

def find_inconsistent_date_formats(df):
    """Detect mixed date representations and invalid date-shaped values."""

    findings = []

    date_patterns = {
        "year-only": r"\d{4}",
        "year-month": r"\d{4}[-/]\d{1,2}",
        "year-first": r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}",
        "numeric-date": r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}",
        "month-year": (
            r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
            r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
            r"Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
            r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
            r"\s+\d{4}"
        ),
        "day-month-year": (
            r"\d{1,2}[-\s]+"
            r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
            r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
            r"Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
            r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
            r"[-,\s]+\d{4}"
        ),
        "month-day-year": (
            r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
            r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
            r"Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
            r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
            r"\s+\d{1,2},?\s+\d{4}"
        )
    }

    compiled_patterns = {
        name: re.compile(r"^\s*(?:" + pattern + r")\s*$", re.I)
        for name, pattern in date_patterns.items()
    }

    month_names = (
        r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
        r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
        r"Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
        r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
    )

    for col in df.columns:
        series = df[col]

        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        ):
            continue

        values = series.dropna().astype(str).str.strip()
        values = values[values.ne("")]

        if len(values) < 2:
            continue

        format_labels = values.map(
            lambda value: next(
                (
                    name for name, pattern in compiled_patterns.items()
                    if pattern.fullmatch(value)
                ),
                None
            )
        )

        recognized = format_labels.notna()
        recognized_values = values[recognized]
        recognized_formats = format_labels[recognized]

        if len(recognized_values) < 2:
            continue

        format_counts = recognized_formats.value_counts()
        non_dominant_count = int(
            len(recognized_formats) - format_counts.iloc[0]
        )

        # Check full date values for impossible dates.
        full_date_patterns = {
            "year-first": (
                r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}"
            ),
            "numeric-date": (
                r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}"
            ),
            "day-month-year": (
                r"\d{1,2}[-\s]+" + month_names +
                r"[-,\s]+\d{4}"
            ),
            "month-day-year": (
                month_names + r"\s+\d{1,2},?\s+\d{4}"
            )
        }

        full_date_regex = re.compile(
            r"^\s*(?:" + "|".join(
                "(?:" + p + ")" for p in full_date_patterns.values()
            ) + r")\s*$",
            re.IGNORECASE
        )

        full_date_values = values[
            values.str.match(full_date_regex, na=False)
        ]

        invalid_date_count = 0

        for value in full_date_values:
            parsed = pd.to_datetime(
                value, format="mixed", errors="coerce"
            )

            # Numeric dates may be day-first or month-first.
            if pd.isna(parsed) and re.fullmatch(
                full_date_patterns["numeric-date"],
                value.strip(),
                flags=re.IGNORECASE
            ):
                parsed = pd.to_datetime(
                    value, format="mixed",
                    dayfirst=True, errors="coerce"
                )

            if pd.isna(parsed):
                invalid_date_count += 1

        if recognized_formats.nunique() > 1:
            findings.append({
                "column": str(col),
                "issue": "Potentially inconsistent date formats",
                "count": non_dominant_count,
                "recognized_values": int(len(recognized_values)),
                "format_count": int(recognized_formats.nunique()),
                "format_breakdown": str(
                    format_counts.to_dict()
                ),
                "invalid_date_count": invalid_date_count,
                "severity": "Low",
                "recommendation": (
                    "Review the different date representations. "
                    "The count is the number of recognized values "
                    "outside the most common format category, not "
                    "the number of invalid dates."
                )
            })

        elif invalid_date_count > 0:
            findings.append({
                "column": str(col),
                "issue": "Potentially invalid date values",
                "count": invalid_date_count,
                "recognized_values": int(len(recognized_values)),
                "format_count": int(recognized_formats.nunique()),
                "format_breakdown": str(
                    format_counts.to_dict()
                ),
                "invalid_date_count": invalid_date_count,
                "severity": "Medium",
                "recommendation": (
                    "Review date-shaped values that could not be "
                    "parsed as valid dates."
                )
            })

    return findings


In [14]:
import pandas as pd

def find_negative_numeric_values(df):
    """Flag negative values in numeric columns without modifying data."""

    findings = []

    for col in df.columns:
        series = df[col]

        # Check numeric columns only; exclude True/False values.
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        ):
            continue

        negative_count = int((series < 0).sum())

        if negative_count > 0:
            findings.append({
                "column": str(col),
                "issue": "Negative numeric values",
                "count": negative_count,
                "severity": "Low",
                "recommendation": (
                    "Review negative values and confirm whether "
                    "they are valid for this column."
                )
            })

    return findings

In [15]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_numeric_outliers(df, file_path):
    """Detect IQR outliers and generate visualizations."""

    findings = []

    for col in df.columns:
        series = df[col]

        # Check numeric columns only; exclude Boolean columns.
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        ):
            continue

        values = series.dropna()

        if len(values) < 4 or values.nunique() < 2:
            continue

        # Calculate IQR bounds.
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            continue

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (values < lower_bound) |
            (values > upper_bound)
        )

        outliers = values[outlier_mask]

        if outliers.empty:
            continue

        # Store actual values in the report.
        outlier_values = outliers.tolist()

        findings.append({
            "column": str(col),
            "issue": "Potential numeric outliers (IQR)",
            "count": len(outlier_values),
            "severity": "Low",
            "outlier_values": str(outlier_values),
            "lower_bound": round(float(lower_bound), 4),
            "upper_bound": round(float(upper_bound), 4),
            "recommendation": (
                "Review the listed values and confirm whether "
                "they are valid observations. Outliers are not "
                "automatically removed."
            )
        })

        # Generate a box plot for this column.
        plt.figure(figsize=(9, 4))
        plt.boxplot(values, vert=False)

        # Highlight the detected outliers.
        plt.scatter(
            outlier_values,
            [1] * len(outlier_values),
            color="red",
            label="Potential outliers",
            zorder=3
        )

        plt.axvline(
            lower_bound,
            linestyle="--",
            label="Lower IQR bound"
        )
        plt.axvline(
            upper_bound,
            linestyle="--",
            label="Upper IQR bound"
        )

        plt.title(f"Outlier Detection: {col}")
        plt.xlabel(str(col))
        plt.yticks([])
        plt.legend()
        plt.tight_layout()

        # Create a separate folder for this dataset's graphs
        dataset_plots_dir = PLOTS_DIR / file_path.stem
        dataset_plots_dir.mkdir(parents=True, exist_ok=True)

        # Create a safe filename for the column
        safe_col = re.sub(r"[^A-Za-z0-9_-]+", "_", str(col)).strip("_")

        # Save the outlier graph
        plot_file = dataset_plots_dir / f"{safe_col}_outlier_boxplot.png"
        plt.savefig(plot_file, dpi=150, bbox_inches="tight")

        #plt.show()
        plt.close()

    return findings


In [92]:
def generate_eda_summary_and_categorical_plots(df, file_path):
    """
    Generate numeric and categorical EDA summaries and graphs.
    Automatically detects common numeric values stored as text.
    Does not modify the original DataFrame.
    """

    import pandas as pd
    import matplotlib.pyplot as plt
    import re

    report_name = file_path.stem

    dataset_plots_dir = PLOTS_DIR / report_name
    dataset_plots_dir.mkdir(parents=True, exist_ok=True)

    # Work on a copy so the original DataFrame is unchanged
    eda_df = df.copy()

    # ----------------------------------
    # 1. Detect numeric-looking text columns
    # ----------------------------------
    for col in eda_df.select_dtypes(
        include=["object", "string"]
    ).columns:

        values = eda_df[col].astype("string").str.strip()

        # Standard numeric values with common formatting
        cleaned = (
            values
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.replace("€", "", regex=False)
            .str.replace("£", "", regex=False)
            .str.replace("¥", "", regex=False)
            .str.strip()
        )

        # Recognize common large-number units.
        # Multipliers are based on the numeric value represented.
        unit_multipliers = {
          "thousand": 1_000,
          "k": 1_000,
          "lakh": 100_000,
          "lac": 100_000,
          "million": 1_000_000,
          "mn": 1_000_000,
          "crore": 10_000_000,
          "cr": 10_000_000,
          "billion": 1_000_000_000,
          "bn": 1_000_000_000,
          "trillion": 1_000_000_000_000,
          "tn": 1_000_000_000_000
        }


        numeric_values = pd.to_numeric(
            cleaned, errors="coerce"
        ).astype("float64")

        # Extract numeric amount and optional unit.
        extracted = cleaned.str.extract(
          r"^\s*([+-]?\d+(?:\.\d+)?)\s*"
          r"(thousand|million|billion|trillion|lakh|lac|crore|"
          r"mn|bn|tn|cr|k)?\s*$",
          flags=re.IGNORECASE
          )

        amounts = pd.to_numeric(extracted[0], errors="coerce")
        units = extracted[1].str.lower()

        multipliers = units.map(unit_multipliers).fillna(1)

        # Use converted amounts when a unit is present;
        # otherwise preserve the ordinary numeric conversion.
        has_unit = units.notna()
        unit_values = amounts * multipliers

        numeric_values = numeric_values.where(~has_unit, unit_values)


        # Recognize numeric values followed by measurement units
        measurement_pattern = (
            r"^\s*([+-]?\d+(?:\.\d+)?)\s*"
            r"(kmpl|km/kg|mpg|cc|bhp|hp|kw|rpm|nm|"
            r"kms?|cm|mm|kg|g|m|%|years?)\s*$"
        )

        measurement_values = cleaned.str.extract(
            measurement_pattern,
            flags=re.IGNORECASE
        )

        measurement_amounts = pd.to_numeric(
            measurement_values[0],
            errors="coerce"
        )

        numeric_values = numeric_values.fillna(
            measurement_amounts
        )


        # Do not classify a column as numeric if most values
        # are actually non-numeric labels.
        non_empty = values.notna() & values.ne("")

        if non_empty.sum() == 0:
            continue

        conversion_rate = (
        numeric_values[non_empty].notna().mean()
        )

        # Check whether any non-empty values failed numeric conversion

        if conversion_rate >= 0.90:
            eda_df[col] = numeric_values.astype("Float64")
            #print(f"EDA: '{col}' treated as numeric.")


    # Exclude columns that contain no valid values
    empty_columns = eda_df.columns[eda_df.isna().all()].tolist()

    if empty_columns:
      #print("Ignoring completely empty columns:", empty_columns)
      eda_df = eda_df.drop(columns=empty_columns)

    # ----------------------------------
    # 2. Numeric summary statistics
    # ----------------------------------
    numeric_df = eda_df.select_dtypes(include="number")

    # Exclude numeric columns that contain no valid values
    numeric_df = numeric_df.dropna(axis=1, how="all")

    if not numeric_df.empty:
        numeric_stats = numeric_df.describe().T
        numeric_stats["median"] = numeric_df.median()
        numeric_stats["missing_values"] = numeric_df.isna().sum()
    else:
        numeric_stats = pd.DataFrame()

    numeric_stats_file = (
        REPORT_DIR / f"{report_name}_numeric_summary.csv"
    )
    numeric_stats.to_csv(numeric_stats_file)

   #print(f"Numeric summary saved: {numeric_stats_file}")

    # ----------------------------------
    # 3. Numeric histograms
    # ----------------------------------
    for col in numeric_df.columns:
        series = numeric_df[col].dropna()

        if series.empty or series.nunique() < 2:
            continue

        safe_col = re.sub(
            r"[^A-Za-z0-9_-]+", "_", str(col)
        ).strip("_")

        plt.figure(figsize=(8, 5))
        plt.hist(series, bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(str(col))
        plt.ylabel("Frequency")
        plt.tight_layout()

        plot_file = (
            dataset_plots_dir / f"{safe_col}_distribution.png"
        )
        plt.savefig(plot_file, dpi=150, bbox_inches="tight")
        plt.close()

        #print(f"Numeric graph saved: {plot_file}")

    # ----------------------------------
    # 4. Correlation heatmap
    # ----------------------------------
    if len(numeric_df.columns) >= 2:
        correlation = numeric_df.corr()

        fig, ax = plt.subplots(figsize=(10, 8))
        image = ax.imshow(correlation, aspect="auto")

        ax.set_xticks(range(len(correlation.columns)))
        ax.set_xticklabels(
            correlation.columns, rotation=90
        )
        ax.set_yticks(range(len(correlation.index)))
        ax.set_yticklabels(correlation.index)

        ax.set_title("Numeric Feature Correlation")
        fig.colorbar(image, ax=ax)
        plt.tight_layout(rect=[0, 0, 0.85, 1])

        heatmap_file = (
            dataset_plots_dir / "correlation_heatmap.png"
        )
        fig.savefig(heatmap_file, dpi=150, bbox_inches="tight")
        plt.close(fig)

        #print(f"Correlation heatmap saved: {heatmap_file}")

    # ----------------------------------
    # 5. Categorical summary and bar charts
    # ----------------------------------
    categorical_df = eda_df.select_dtypes(
        include=["object", "string", "category", "bool"]
    )

    categorical_rows = []

    for col in categorical_df.columns:
        series = categorical_df[col]
        non_missing = series.dropna()
        value_counts = non_missing.value_counts()

        if not value_counts.empty:
            most_frequent = value_counts.index[0]
            most_frequent_count = int(value_counts.iloc[0])
        else:
            most_frequent = None
            most_frequent_count = 0

        categorical_rows.append({
            "column": col,
            "unique_values": series.nunique(),
            "most_frequent_value": most_frequent,
            "most_frequent_count": most_frequent_count,
            "missing_values": int(series.isna().sum())
        })

        if value_counts.empty:
            continue

        top_categories = value_counts.head(10)

        safe_col = re.sub(
            r"[^A-Za-z0-9_-]+", "_", str(col)
        ).strip("_")

        plt.figure(figsize=(10, 5))
        top_categories.sort_values().plot(kind="barh")
        plt.title(f"Top Categories - {col}")
        plt.xlabel("Number of Records")
        plt.ylabel(str(col))
        #plt.tight()

        plt.subplots_adjust(
          left=0.15,
          right=0.95,
          bottom=0.30,
          top=0.90
        )

        plot_file = (
            dataset_plots_dir
            / f"{safe_col}_category_distribution.png"
        )
        plt.savefig(plot_file, dpi=150, bbox_inches="tight")
        plt.close()

        #print(f"Categorical graph saved: {plot_file}")

    categorical_stats = pd.DataFrame(
        categorical_rows,
        columns=[
            "column",
            "unique_values",
            "most_frequent_value",
            "most_frequent_count",
            "missing_values"
        ]
    )

    categorical_stats_file = (
        REPORT_DIR / f"{report_name}_categorical_summary.csv"
    )
    categorical_stats.to_csv(
        categorical_stats_file, index=False
    )

    #print(f"Categorical summary saved: {categorical_stats_file}")

    return numeric_stats, categorical_stats


In [83]:
import pandas as pd

def run_data_quality_checks(df):
    """
    Identify common data-quality issues in any DataFrame.
    Does not modify or delete the original data.
    """

    results = []

    total_rows = len(df)

    for col in df.columns:
        series = df[col]
        missing_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))

        # 1. Check missing values and completely empty columns
        if total_rows > 0 and missing_count == total_rows:
            results.append({
                "column": col,
                "issue": "Completely empty column",
                "count": missing_count,
                "severity": "High",
                "recommendation":
                    "All values in this column are NaN/null. "
                    "Consider deleting this column."
           })

        elif missing_count > 0:
            results.append({
                "column": col,
                "issue": "Missing values",
                "count": missing_count,
                "severity": (
                    "High" if missing_count / total_rows >= 0.5
                    else "Medium"
                ),
                "recommendation":
                    "Review missing values before deciding how to handle them."
            })

         # 2. Check constant columns
        if total_rows > 0 and unique_count == 1 :
            results.append({
                "column": col,
                "issue": "Constant",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column provides useful information."
            })


         # 3. Potential numeric data stored as text
        if pd.api.types.is_object_dtype(series) or \
           pd.api.types.is_string_dtype(series):

            non_missing = series.dropna().astype("string").str.strip()

            if len(non_missing) > 0:

                # Remove common formatting and measurement units
                cleaned = (
                    non_missing
                    .str.replace(",", "", regex=False)
                    .str.replace("₹", "", regex=False)
                    .str.replace("$", "", regex=False)
                    .str.replace("%", "", regex=False)
                    .str.replace("€", "", regex=False)
                    .str.replace("£", "", regex=False)
                    .str.replace("¥", "", regex=False)
                    .str.strip()
                )

                # Extract numeric amounts with optional units

                extracted = cleaned.str.extract(
                    r"^\s*([+-]?\d+(?:\.\d+)?)\s*"
                    r"(kmpl|km/kg|mpg|cc|bhp|hp|kw|rpm|nm|"
                    r"kms?|cm|mm|kg|g|m|thousand|million|"
                    r"billion|trillion|lakh|lac|crore|"
                    r"mn|bn|tn|cr|k|years?)?\s*$",
                    flags=re.IGNORECASE
                )


                converted = pd.to_numeric(
                    extracted[0],
                    errors="coerce"
                )

                numeric_ratio = converted.notna().mean()

                if numeric_ratio >= 0.8:
                    results.append({
                        "column": col,
                        "issue": "Possible numeric data stored as text",
                        "count": int(converted.notna().sum()),
                        "severity": "Medium",
                        "recommendation":
                            "Review the values and convert to numeric if appropriate."
                    })

        # 4. High-cardinality columns
        if total_rows > 0 and unique_count / total_rows >= 0.9:
            results.append({
                "column": col,
                "issue": "High cardinality",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column contains IDs or mostly unique values."
            })

    # 5. Duplicate rows (dataset-level issue)
    duplicate_count = int(df.duplicated().sum())

    # 6. Identical columns
    results.extend(find_identical_columns(df))

    if duplicate_count > 0:
        results.append({
            "column": "(entire dataset)",
            "issue": "Duplicate rows",
            "count": duplicate_count,
            "severity": "Medium",
            "recommendation":
                "Review duplicate records before removing them."
        })

    # 7. Potentially equivalent columns
    results.extend(find_near_duplicate_columns(df))

    # 8. Potentially inconsistent date formats
    results.extend(find_inconsistent_date_formats(df))

    # 9. Potentially negative numeric values
    results.extend(find_negative_numeric_values(df))

    return pd.DataFrame(
        results,
        columns=[
            "column", "issue", "count",
            "severity", "recommendation"
        ]
    )


In [93]:
import json
import pandas as pd

def process_dataset(file_path):
    file_path = Path(file_path)

    print(f"\nProcessing: {file_path.name}")

    # 1. Load dataset
    df = load_dataset(file_path)

    # 2. Profile dataset
    profile = profile_dataset(df)

    # 3. Review suspicious Unnamed columns
    unnamed_report = review_unnamed_columns(df)

    # 4. Create missing-value report
    missing_report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (df.isna().mean() * 100).round(2)
    })

    missing_report = missing_report[
        missing_report["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)

    # 5. Run data-quality checks
    quality_report = run_data_quality_checks(df)

    # 6. Check for inconsistent date formats
    date_format_findings = find_inconsistent_date_formats(df)
    date_format_report = pd.DataFrame(date_format_findings)

    # 7. Generate the separate outlier report
    outlier_findings = find_numeric_outliers(df, file_path)

    outlier_report = pd.DataFrame(
       outlier_findings,
       columns=[
        "column",
        "issue",
        "count",
        "severity",
        "recommendation",
        "outlier_values",
        "lower_bound",
        "upper_bound"
      ]
    )


    # 8. Save reports using the dataset's filename
    report_name = file_path.stem

    with open(REPORT_DIR / f"{report_name}_profile.json", "w") as f:
        json.dump(profile, f, indent=4, default=str)

    missing_report.to_csv(
        REPORT_DIR / f"{report_name}_missing_values.csv"
    )

    unnamed_report.to_csv(
        REPORT_DIR / f"{report_name}_column_review.csv",
        index=False
    )

    quality_report.to_csv(
        REPORT_DIR / f"{report_name}_data_quality_report.csv",
        index=False
    )

    date_format_report.to_csv(
        REPORT_DIR / f"{file_path.stem}_date_format_report.csv",
        index=False
    )


    outlier_report.to_csv(
        REPORT_DIR / f"{report_name}_outlier_report.csv",
        index=False
    )

    # 9. Generate EDA summaries and plots
    numeric_stats, categorical_stats = (
        generate_eda_summary_and_categorical_plots(
            df, file_path
        )
    )

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    return df, profile, missing_report, unnamed_report
